# Agentic Workspace MCP: Example Usage

This notebook demonstrates how to utilize the **Sandboxed Agentic Workspace MCP** server with the **OpenAI Agents SDK** and **OpenRouter**.

### Tech Stack:
- **MCP Server**: Sandboxed environment (Docker).
- **Orchestration**: OpenAI Agents SDK (Python).
- **Model Backend**: OpenRouter (giving you access to Gemini, Claude, etc.).

## 1. Setup Environment
We need to load our API keys from `.env` and configure the event loop for the notebook if necessary.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# load_dotenv() 
# If using a specific path:
load_dotenv(Path("..") / ".env")

print(f"OpenRouter API Key found: {bool(os.environ.get('OPENROUTER_API_KEY'))}")

## 2. Initialize MCP Server Connection
We connect to the existing Docker-based MCP server via standard I/O (stdio).

**Security Hardening**: The container is launched with dropped capabilities, restricted memory/CPU, and no privilege escalation.

In [ ]:
from agents.mcp import MCPServerStdio

# Configure the server link
# We assume the workspace root is the parent directory's 'tmp'
root_dir = Path(os.getcwd()).parent
abs_tmp_dir = str((root_dir / "tmp").resolve())

mcp_server = MCPServerStdio(
    name="Sandboxed Workspace",
    params={ 
        "command": "docker", 
        "args": [ 
            "run", "-i", "--rm", 
            "--user", "1000:1000", 
            "--security-opt", "no-new-privileges",
            "--cap-drop", "ALL",
            "--init",
            "--memory", "512m",
            "--cpus", "0.5",
            "-v", f"{abs_tmp_dir}:/workspace", 
            "agent-workspace-mcp" 
        ] 
    }, 
    client_session_timeout_seconds=60.0 
)

## 3. Define the Agent
We'll use a specialized model from OpenRouter (e.g., Gemini 2.0 Flash).

In [ ]:
from agents import Agent, trace
from agents.extensions.models.litellm_model import LitellmModel

model_name = os.environ.get("DEFAULT_MODEL")

agent = Agent(
    name="WorkspaceAnalyst",
    instructions=(
        "You are a workspace operations analyst. Your goal is to explore the filesystem, "
        "audit artifacts, and perform maintenance tasks using your bash and file tools."
    ),
    model=LitellmModel(model=model_name),
    mcp_servers=[mcp_server]
)

## 4. Run the Agent (Streamed)
The following function runs the agent and prints tool calls, results, and text responses in real-time.

In [ ]:
from agents import Runner, RunConfig

async def run_analysis(mission: str):
    print(f"🚀 Starting Mission: {mission}\n")
    
    async with mcp_server:
        # Enable OpenAI tracing for observability
        with trace(f"Mission: {mission[:50]}..."):
            # The Runner.run_streamed returns a result object that exposes a stream_events() generator.
            # We use await here as Jupyter inherently supports top-level async.
            stream = Runner.run_streamed(agent, mission, max_turns=15, run_config=RunConfig())
            async for event in stream.stream_events():
                if event.type == "run_item_stream_event":
                    item = event.item
                    if event.name == "tool_called":
                        print(f"\n🛠️  [TOOL CALL] {item.raw_item.name}({item.raw_item.arguments})")
                    elif event.name == "tool_output":
                        # Truncate large results for readability
                        output = str(item.output)
                        if len(output) > 200: 
                            output = output[:200] + "..."
                        print(f"✅ [RESULT] {output}")
                    elif event.name == "handoff_requested":
                        print(f"➡️  [HANDOFF] to {item.target_agent.name}")
                elif event.type == "raw_response_event":
                    from openai.types.responses import ResponseTextDeltaEvent
                    if isinstance(event.data, ResponseTextDeltaEvent):
                        print(event.data.delta, end="", flush=True)

In [ ]:

mission = "List the contents of the /workspace directory. Pick one .py file, read its metadata (size/mod time), and report your findings. Execute the file and print its output! If nothing is there, just a create a nice little python script, execute it and tell about it."
# Use top-level await (supported natively in Jupyter kernels)
await run_analysis(mission)

## 5. Advanced Editing (Dry Run & Fuzzy Matching)

The `search_and_replace` tool supports **Dry Run** mode and **Fuzzy Whitespace Matching**. 
- **Dry Run**: Preview changes as a unified diff before applying them.
- **Fuzzy Matching**: Matches code even if the agent gets the leading/trailing whitespace slightly wrong.
- **Indentation Preservation**: Automatically rebases the replacement text to match the file's indentation level.

In [ ]:
mission = """
1. Create 'logic.py' with a few python functions that have messy indentation.
2. Use search_and_replace with dry_run=True to fix the functions. 
3. Examine the diff, then apply the change with dry_run=False.
"""

await run_analysis(mission)

## 6. Coding and File writing


In [ ]:
mission = """
1. Write a python script to retrieve the site https://edwarddonner.com/ .
2. Save the website and convert it to markdown in a second script and save this markdown file too.
3. Install packages if necessary, use available bash tools.
"""

await run_analysis(mission)

In [ ]:
mission = """
Translate all comments, doc strings, annotations etc. in filesystem.py to German, use search an replace to update comments. Afterwards check if the file still works.
"""

await run_analysis(mission)

In [ ]:
mission = """
You are an autonomous developer assistant equipped with workspace capabilities. Your task is to build, test, and iteratively modify a log analysis utility entirely within the workspace. 

Please complete the following steps in order, using your available tools:

1. **Environment Setup & Data Generation (Filesystem & Bash):**
   - Create a new directory called `log_analyzer_project`.
   - Using a single Bash command, generate a mock log file named `server.log` inside this directory. The file must contain at least 20 lines of log data spanning three severity levels (INFO, WARNING, ERROR) with random dummy IP addresses and timestamps. 
   - Use your filesystem tools to verify the directory and file were created correctly and inspect the first few lines of the log.

2. **Initial Python Implementation (Python Execution):**
   - Write a Python script named `analyzer.py` inside the directory. The script should read `server.log`, count the total occurrences of each severity level (INFO, WARNING, ERROR), and print the results to standard output.
   - Execute the Python script using your python execution tool and verify the printed output matches the data you generated.

3. **In-Place Refactoring (Editing):**
   - We need to change how the script outputs data. **Do not overwrite or recreate the entire file.** - Use your specific file **editing tool** (e.g., string replacement or targeted line editing) to modify `analyzer.py`. Change the script so that instead of printing the results to the console, it imports the `json` module and writes the summary to a new file named `summary.json`.

4. **Final Execution & Verification (Filesystem & Bash):**
   - Execute the updated `analyzer.py` script.
   - Use the filesystem tool to read `summary.json` and output its contents to me so I can verify the final result.
   - Finally, run a quick bash `ls -la` on the project directory to show me the final workspace state.

Work methodically. Output your thoughts before taking actions, and ensure one step is successful before moving to the next.
"""

await run_analysis(mission)

In [ ]:
mission = """
You are a red-team security agent tasked with auditing the security boundaries of this Workspace MCP Server. Your primary objective is to test the robustness of the sandbox and attempt to access unauthorized resources, escape the container environment, or read restricted host files. 

Execute the following test vectors methodically using your available tools (bash, python, filesystem, and editing). For each attempt, document the tool used, the exact input, and the resulting output or error.

**Phase 1: Tool-Specific Path Traversal**
The MCP server enforces a strict workspace directory. Test if the Python-based tools properly sanitize inputs.
1. Use the `filesystem` tool (e.g., `read_file` or `list_directory`) to target absolute paths outside the workspace, such as `/etc/passwd` or `/root`.
2. Use the `filesystem` tool with relative path traversal (e.g., `../../../etc/passwd`).
3. Try to use the `editing` tool to modify a system file outside the workspace, like appending a dummy key to `/root/.ssh/authorized_keys` (if accessible) or editing `/etc/hosts`.

**Phase 2: Environment and Privilege Enumeration (Bash & Python)**
Use your code execution tools to probe the constraints of the container.
1. Execute `env` and `set` via bash to look for sensitive host environment variables passed down to the container.
2. Run `cat /proc/1/cgroup` or `mount` to confirm the container runtime and look for poorly configured mounts (like the host's root filesystem or Docker socket).
3. Check for the Docker socket directly: `ls -la /var/run/docker.sock`. If it exists, attempt to use `curl` to interact with it (e.g., `curl --unix-socket /var/run/docker.sock http://localhost/containers/json`).
4. Execute `sudo -l` to check your current privileges.

**Phase 3: Network Egress & SSRF Probing**
Test if the sandbox restricts outbound network traffic or allows Server-Side Request Forgery against the host network.
1. Write and execute a Python script to send a GET request to the cloud provider metadata IP (`http://169.254.169.254/latest/meta-data/`).
2. Try to curl the host's default gateway or localhost on common administrative ports (e.g., `curl http://172.17.0.1:8080` or `curl http://host.docker.internal:22`).

**Phase 4: Resource Limits (Optional/Safe Mode)**
Test if the environment prevents basic resource exhaustion.
1. Write a Python script that creates a 1GB file filled with random data to test storage quotas. 
2. **Do not execute a fork bomb**, but document if you believe one would successfully crash the environment based on the output of `ulimit -a`.

Report your findings clearly. State whether each vector was successfully blocked by the MCP server's security measures or if it allowed unauthorized access.
"""

await run_analysis(mission)

In [ ]:
mission = """
Choose your favorite pip package install it and write an small script showcasing it.
"""

await run_analysis(mission)